# Fase 3 — Memória de Curto e Longo Prazo: o Agente Vira Chatbot

Um agente que raciocina e age, mas esquece tudo a cada mensagem, não sustenta uma sessão de planejamento real. Nesta fase o agente da Fase 2 ganha:

1. **memória de curto prazo** — `ConversationBufferMemory` com janela deslizante + resumo;
2. **memória de longo prazo** — semântica e episódica, persistente entre sessões, com busca vetorial;
3. **reconhecimento de intenção e extração de entidades**, com um estado da sessão e fluxos de conversa que pedem esclarecimento quando falta informação.

In [1]:
import sys, pathlib, json, warnings
RAIZ = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "fabrica_agentica").is_dir())
sys.path.insert(0, str(RAIZ))
from fabrica_agentica.config import descrever_ambiente
print(descrever_ambiente())

Provedor de LLM : anthropic
APIs externas   : rede (ao vivo)
Embeddings      : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Modelo NLI      : MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli
LangSmith       : ligado
Repositório GH  : DriccoC/fabrica-agentica-sandbox


## 1. Estratégia de memória (antes de implementar)

### 1.1 Curto × longo prazo no domínio, pela taxonomia CoALA

O CoALA (*Cognitive Architectures for Language Agents*, Sumers et al., 2023) separa a **memória de trabalho** (o contexto da interação corrente) das memórias de longo prazo **semântica** (conhecimento sobre o mundo), **episódica** (experiências passadas) e **procedural** (como fazer as coisas).

| CoALA | Horizonte | No Tech Lead Agêntico | Implementação |
|---|---|---|---|
| **Trabalho** (curto prazo) | a sessão | demanda em discussão, projeto, sprint, pessoas retiradas, plano proposto, biblioteca sugerida | `MemoriaJanelaResumo` (subclasse de `ConversationBufferMemory`) + estado de entidades da sessão |
| **Semântica** (longo prazo) | permanente | "Adriano é tech lead do Portal", "prefere histórias de até 5 pontos", "o time usa pytest" | `MemoriaLongoPrazo` tipo `semantica` (SQLite + embeddings) |
| **Episódica** (longo prazo) | permanente | "em 05/11 planejamos o login com Google: 11 pontos, sem o Bruno o saldo caiu para 9" | `MemoriaLongoPrazo` tipo `episodica`, gravada ao encerrar a sessão |
| **Procedural** | permanente, versionada | Definição de Pronto, fórmula de capacidade, regras de CVSS, fluxo de planejamento | system prompt PTCF e ferramentas *hardcoded* (não muda com a conversa) |

**O que NÃO vai para a memória de longo prazo:** regras das normas (já estão na base de conhecimento FAISS) e dados que as ferramentas recalculam (saldo atual da sprint, versão mais recente de uma biblioteca, vulnerabilidades). Guardá-los criaria memórias que envelhecem e contradizem a fonte de verdade. A memória guarda **decisões e preferências**; os **fatos voláteis** vêm sempre das ferramentas.

### 1.2 Gestão da janela de contexto

O agente já consome ~40 mil tokens de entrada numa demanda complexa (prompt PTCF, *few-shot*, esquemas de 12 ferramentas e observações). A conversa não pode crescer sem limite. Estratégia em três camadas:

1. **Janela deslizante:** as últimas *N* mensagens vão literalmente ao prompt (aqui *N* = 4, para a demonstração mostrar o efeito; em produção, 6 a 10).
2. **Sumarização progressiva:** mensagens que saem da janela são incorporadas a um resumo (LLM auxiliar barato), com instrução de preservar projeto, sprint, pessoas, números e decisões.
3. **Busca vetorial na memória de longo prazo:** a cada turno, recuperamos só as *k* = 4 memórias mais relevantes para a mensagem e o estado da sessão (similaridade de cosseno ≥ 0,35). Ao gravar, se já existir uma memória quase idêntica (cosseno ≥ 0,90), ela é **atualizada** direto; senão, as 3 memórias mais parecidas vão a um LLM auxiliar que decide **adicionar** ou **atualizar** uma delas (a mesma ideia de ADD/UPDATE do Mem0) — necessário porque o modelo de embeddings pequeno dá só 0,44 de similaridade entre "prefere histórias de até 5 pontos" e "prefere histórias de até 3 pontos".
4. **Recência + relevância:** o episódio mais recente sempre acompanha a recuperação, porque perguntas como "o que fizemos da última vez?" têm baixa similaridade com o texto do episódio.

### 1.3 Comparação de ferramentas especializadas de memória

| | **LangMem** (LangChain) | **Mem0** | **Letta** (ex-MemGPT) |
|---|---|---|---|
| Arquitetura | biblioteca de primitivas: extração de memórias por LLM, *namespaces*, gravação no `Store` do LangGraph (semântica, episódica, procedural com otimização de prompt) | camada de memória como serviço (SDK + API/hospedado): extrai, deduplica e atualiza fatos com LLM; armazenamento vetorial e opcionalmente em grafo | "sistema operacional" de memória: o agente gerencia a própria memória em níveis (núcleo no contexto, arquivo e recall fora dele) por meio de chamadas de função |
| Quem decide o que lembrar | o desenvolvedor configura extratores (em segundo plano ou no fluxo) | o serviço, automaticamente | o próprio agente, via ferramentas de memória |
| Integração com LangChain/LangGraph | nativa | via SDK/integração | servidor próprio; agente roda dentro do Letta |
| Caso de uso típico | agentes LangGraph que precisam de memória de longo prazo sem infraestrutura nova | assistentes multiusuário que precisam de personalização entre aplicações | agentes de longa duração com contexto virtualmente ilimitado |
| Custo/complexidade aqui | baixo, mas acopla ao `Store` do LangGraph | serviço externo, mais uma chave/dependência | alto: muda o *runtime* do agente |

**Decisão:** nenhuma entra como dependência nesta disciplina. Implementamos uma memória própria e pequena (~150 linhas), **com o mesmo desenho do LangMem** (extração por LLM → fatos tipados semântica/episódica → *namespace* por usuário → busca vetorial → atualização em vez de duplicação). Motivos: (1) o executor exigido pela rubrica é o `AgentExecutor` clássico, e o LangMem é pensado para o `Store` do LangGraph; (2) Mem0 hospedado adicionaria uma credencial e enviaria dados a terceiros, contra o requisito de reprodutibilidade sem credenciais; (3) Letta substituiria o executor. Na próxima disciplina, com LangGraph, a migração natural é trocar `MemoriaLongoPrazo` pelo `Store` + LangMem, mantendo os mesmos tipos de memória.

## 2. Reconhecimento de intenção e extração de entidades

Cada mensagem passa por um classificador (Claude Haiku com saída estruturada `IntencaoEntidades`). As entidades se acumulam num **estado da sessão**: quando o usuário diz "e se tirarmos o Bruno?", o projeto e a sprint vêm do estado, não da mensagem.

In [2]:
from fabrica_agentica.memoria import intencao as fluxo

estado = {}
mensagens_teste = [
    "Oi! Sou o Adriano, tech lead do Portal.",
    "Quero login com Google na próxima sprint.",
    "E se tirarmos o Bruno da sprint 13?",
    "A authlib 1.3.0 tem vulnerabilidade?",
    "Posso fazer deploy na sexta à tarde?",
]
print(f"{'mensagem':45} | {'intenção':18} | entidades da mensagem -> estado acumulado")
for m in mensagens_teste:
    r = fluxo.reconhecer(m, estado)
    estado = fluxo.atualizar_estado(estado, r)
    ents = {k: v for k, v in r.model_dump().items() if k != "intencao" and v}
    print(f"{m[:45]:45} | {r.intencao:18} | {ents} -> {estado}")

mensagem                                      | intenção           | entidades da mensagem -> estado acumulado


Oi! Sou o Adriano, tech lead do Portal.       | conversa_geral     | {'projeto': 'Portal', 'pessoa': 'Adriano'} -> {'projeto': 'Portal', 'pessoa': 'Adriano'}


Quero login com Google na próxima sprint.     | planejar_demanda   | {} -> {'projeto': 'Portal', 'pessoa': 'Adriano'}


E se tirarmos o Bruno da sprint 13?           | planejar_demanda   | {'sprint': 13, 'pessoa': 'Bruno'} -> {'projeto': 'Portal', 'pessoa': 'Bruno', 'sprint': 13}


A authlib 1.3.0 tem vulnerabilidade?          | checar_dependencia | {'pacote': 'authlib', 'versao': '1.3.0'} -> {'projeto': 'Portal', 'pessoa': 'Bruno', 'sprint': 13, 'pacote': 'authlib', 'versao': '1.3.0'}


Posso fazer deploy na sexta à tarde?          | consultar_padrao   | {} -> {'projeto': 'Portal', 'pessoa': 'Bruno', 'sprint': 13, 'pacote': 'authlib', 'versao': '1.3.0'}


**Fluxos de conversa:** cada intenção tem entidades obrigatórias. Planejar uma demanda exige saber o projeto; checar uma dependência exige o pacote. Se faltar e não estiver no estado, o chatbot pergunta antes de acionar o agente (sem gastar uma execução cara à toa):

In [3]:
print(json.dumps(fluxo.FLUXOS, ensure_ascii=False, indent=1))
print("Lacunas para 'planejar_demanda' sem projeto:", fluxo.lacunas("planejar_demanda", {}))
print("Lacunas com projeto no estado:", fluxo.lacunas("planejar_demanda", {"projeto": "Portal"}))

{
 "planejar_demanda": {
  "projeto": "Para qual projeto é essa demanda: Portal, PagaFacil ou AppCampo?"
 },
 "consultar_backlog": {
  "projeto": "De qual projeto você quer saber: Portal, PagaFacil ou AppCampo?"
 },
 "checar_dependencia": {
  "pacote": "Qual biblioteca (pacote) você quer que eu verifique?"
 },
 "consultar_padrao": {},
 "conversa_geral": {}
}
Lacunas para 'planejar_demanda' sem projeto: ['Para qual projeto é essa demanda: Portal, PagaFacil ou AppCampo?']
Lacunas com projeto no estado: []


## 3. Chatbot com memória: sessão 1 (conversa real de múltiplos turnos)

Limpamos a memória de longo prazo do usuário para a demonstração começar do zero. A janela de curto prazo é de 4 mensagens (2 turnos), para que a sumarização apareça já nesta conversa.

In [4]:
from fabrica_agentica.agentes.chatbot import ChatbotTechLead
from fabrica_agentica.memoria.longo_prazo import MemoriaLongoPrazo
from fabrica_agentica.observabilidade.callbacks import RegistroExecucao

MemoriaLongoPrazo(escopo="adriano").esquecer_tudo()
log = RegistroExecucao(sessao="fase3")
bot = ChatbotTechLead(usuario="adriano", janela=4, callbacks=[log])

def mostrar(turno):
    print(f"\n🧑 {turno.mensagem}")
    print(f"   intenção: {turno.intencao} | estado da sessão: {turno.estado_sessao}")
    if turno.memorias_usadas:
        print(f"   memórias de longo prazo recuperadas: {turno.memorias_usadas}")
    if turno.trajetoria:
        print(f"   ferramentas: {[p['ferramenta'] for p in turno.trajetoria]}")
    print(f"🤖 {turno.resposta}")
    if turno.memorias_gravadas:
        print(f"   💾 memórias gravadas: {[m.get('conteudo', m.get('depois')) for m in turno.memorias_gravadas]}")

mostrar(bot.conversar("Oi! Sou o Adriano, tech lead do Portal. Prefiro histórias de no máximo 5 pontos, e o time sempre usa pytest."))

/Users/adrianocarvalho/Documents/_MBA/Agentes_Cognitivos/projeto_fabrica_agentica/.venv/lib/python3.12/site-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


W1105 17:14:57.456000 73806 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


🧑 Oi! Sou o Adriano, tech lead do Portal. Prefiro histórias de no máximo 5 pontos, e o time sempre usa pytest.
   intenção: conversa_geral | estado da sessão: {'projeto': 'Portal', 'pessoa': 'Adriano'}
🤖 Oi, Adriano! Anotei o que você disse sobre o Portal:

- **Tamanho das histórias:** no máximo 5 pontos. Vou quebrar demandas maiores em histórias menores.
- **Testes:** o time usa pytest. Vou colocar isso nas tarefas e nos critérios de aceite, e verificar a versão e as vulnerabilidades dele quando for relevante.

Isso vale para esta conversa. Não tenho memória de sessões anteriores sobre você.

Posso ajudar com:
- **Planejar uma demanda nova:** capacidade da sprint, normas aplicáveis, bibliotecas, issues relacionadas, e depois o épico com histórias e tarefas.
- **Backlog do Portal:** consultas sobre histórias, bugs e sprints.
- **Manual de Engenharia:** dúvidas sobre MAN-01 a MAN-07.
- **Bibliotecas:** versões no PyPI e vulnerabilidades.
- **Capacidade:** saldo de pontos de uma sprint,

In [5]:
# Turno 2: o projeto vem do estado da sessão; a preferência (<= 5 pontos) vem do turno anterior
mostrar(bot.conversar("Quero login com Google na próxima sprint."))


🧑 Quero login com Google na próxima sprint.
   intenção: planejar_demanda | estado da sessão: {'projeto': 'Portal', 'pessoa': 'Adriano', 'pacote': 'pytest'}
   memórias de longo prazo recuperadas: ['Adriano é tech lead do Portal e prefere histórias de no máximo 5 pontos.']
   ferramentas: ['calcular_capacidade_sprint', 'buscar_manual_engenharia', 'buscar_issues_github', 'buscar_issues_github', 'consultar_pypi', 'consultar_pypi', 'consultar_vulnerabilidades', 'consultar_vulnerabilidades', 'decompor_demanda']
🤖 ## Conclusão

O login com Google cabe na próxima sprint do Portal (sprint 13). O plano tem 3 histórias e 11 pontos, e o saldo é de 14. Sobrariam 3 pontos. Todas as histórias têm no máximo 5 pontos, como você prefere.

Ainda não registrei nada no backlog. Se quiser, é só pedir.

## Plano: épico "Login com Google no Portal"

| # | História | Pontos |
|---|---|---|
| 1 | Entrar no Portal com a conta Google (OIDC com código de autorização e PKCE S256, rejeitando fluxos proibidos) | 5

In [6]:
# Turno 3: "o plano" e "a sprint" só fazem sentido com o contexto dos turnos anteriores
mostrar(bot.conversar("E se tirarmos o Bruno da sprint, o plano ainda cabe?"))


🧑 E se tirarmos o Bruno da sprint, o plano ainda cabe?
   intenção: planejar_demanda | estado da sessão: {'projeto': 'Portal', 'pessoa': 'Bruno', 'pacote': 'pytest', 'sprint': 13}
   memórias de longo prazo recuperadas: ["O épico 'Login com Google no Portal' foi planejado para a sprint 13 com 3 histórias e 11 pontos, sem registro no backlog ainda."]
   ferramentas: ['calcular_capacidade_sprint']
🤖 ## Conclusão

**Sim, o plano ainda cabe sem o Bruno.** O saldo cai de 14 para 9 pontos (`calcular_capacidade_sprint`, MAN-06 §3). O plano tem 11 pontos, mas desses, 11 já estão comprometidos na sprint, então a conta precisa de um esclarecimento (veja abaixo).

## Atenção: o 11 já está no comprometido?

Na simulação, o resultado mostra 11 pontos comprometidos e saldo de 9. Isso bate com a conta anterior, em que o saldo era 14 com os mesmos 11 comprometidos. Como o plano do Login com Google ainda não foi registrado, esses 11 são outras histórias da sprint, e não o nosso plano. Por isso:

- **S

In [7]:
# Turno 4: "a biblioteca que você sugeriu" é resolvida pela memória de curto prazo
mostrar(bot.conversar("A biblioteca que você sugeriu é segura na versão que usamos hoje em outro serviço, a 1.3.0?"))


🧑 A biblioteca que você sugeriu é segura na versão que usamos hoje em outro serviço, a 1.3.0?
   intenção: checar_dependencia | estado da sessão: {'projeto': 'Portal', 'pessoa': 'Bruno', 'pacote': 'authlib', 'sprint': 13, 'versao': '1.3.0'}
   memórias de longo prazo recuperadas: ['O time do Portal usa pytest como framework de testes.', '(sessão mais recente) Em análise: retirar o Bruno da sprint para encaixar o plano do Login com Google, que não cabe sem ele; a decisão ainda não foi tomada.']
   ferramentas: ['consultar_vulnerabilidades', 'comparar_versoes']
🤖 ## Conclusão

**Não, a authlib 1.3.0 não é segura.** Para o Login com Google, ela está bloqueada pelo MAN-04. Use a 1.9.5.

Presumi que você fala da authlib, que é a biblioteca do plano. Se for o pytest, me avise e eu verifico.

## Fundamentos

- **Vulnerabilidades (OSV):** a 1.3.0 tem 34 vulnerabilidades conhecidas, e a maior tem CVSS 9,1 (crítico). Esse valor passa do limite de 7,0 que bloqueia a adoção (MAN-04). As principai

### Janela deslizante + resumo em ação

Com janela de 4 mensagens, os turnos 1 e 2 já saíram da janela literal e estão no **resumo**. O turno seguinte testa se o contexto sobreviveu à compressão.

In [8]:
print("RESUMO DOS TURNOS ANTIGOS:\n", bot.memoria_curta.resumo)
print(f"\nMensagens literais na janela: {len(bot.memoria_curta.chat_memory.messages) - bot.memoria_curta.resumidas}"
      f" | no buffer completo (auditoria): {len(bot.memoria_curta.chat_memory.messages)}")

RESUMO DOS TURNOS ANTIGOS:
 Sessão com Adriano, tech lead do Portal, sobre login com Google na sprint 13, a próxima sprint planejada (09/11 a 20/11, 9 dias úteis, com feriado em 20/11). A capacidade líquida é de 25 pontos, com 11 já comprometidos e saldo de 14, e o Bruno tem 7 dias disponíveis. O plano do épico "Login com Google no Portal" tem 3 histórias, somando 11 pontos, com no máximo 5 cada, usando OIDC com PKCE, authlib 1.9.5 e pytest 9.2.0 (sem vulnerabilidades conhecidas), e exige MFA para operações sensíveis e administradores. Diretrizes válidas só nesta conversa: histórias de até 5 pontos e pytest nas tarefas e critérios de aceite; nenhuma pessoa foi retirada e não há issues duplicadas no GitHub. Pendências: confirmar se o escopo é só pessoa física (B2B exige federação própria), decidir o registro das histórias na sprint 13 com abertura de issues (ainda não feito) e monitorar a authlib 1.9.5, lançada em 05/11/2026.

Mensagens literais na janela: 4 | no buffer completo (audito

In [9]:
# Turno 5: depende de informações que agora só existem no resumo (preferência do turno 1, plano do turno 2)
mostrar(bot.conversar("Resuma o que decidimos até agora e confirme se o plano respeita a minha preferência de tamanho de história."))


🧑 Resuma o que decidimos até agora e confirme se o plano respeita a minha preferência de tamanho de história.
   intenção: conversa_geral | estado da sessão: {'projeto': 'Portal', 'pessoa': 'Bruno', 'pacote': 'authlib', 'sprint': 13, 'versao': '1.3.0'}
   memórias de longo prazo recuperadas: ['Foi proposta a atualização da authlib 1.3.0 para 1.9.5 no outro serviço que ainda usa a 1.3.0, como história separada na sprint 13.']
🤖 ## Conclusão

O plano respeita a sua preferência: as 3 histórias têm no máximo 5 pontos cada (5, 3 e 3, total de 11). O pytest também está previsto nas tarefas e nos critérios de aceite. Não refiz nenhuma consulta neste turno. É um resumo do que já apurei.

## O que decidimos até agora

- **Escopo:** épico "Login com Google no Portal", na sprint 13 (09/11 a 20/11, 9 dias úteis, com feriado em 20/11).
- **Tecnologia:** OIDC com PKCE, authlib 1.9.5 e pytest 9.2.0, sem vulnerabilidades conhecidas (PyPI e OSV).
- **Segurança:** MFA obrigatório para operações sensíve

In [10]:
episodio = bot.encerrar_sessao()
print("Episódio gravado na memória de longo prazo:\n", episodio)
print("\nMEMÓRIA DE LONGO PRAZO DO USUÁRIO:")
for m in MemoriaLongoPrazo(escopo="adriano").listar():
    print(f" - [{m['tipo']}] {m['conteudo']}")

Episódio gravado na memória de longo prazo:
 {'acao': 'atualizada', 'memoria_id': 91, 'antes': "O épico 'Login com Google no Portal' foi planejado para a sprint 13, usando OIDC com PKCE via authlib.", 'depois': 'Em 05/11/2026, o épico Login com Google no Portal foi planejado para a sprint 13 (09/11 a 20/11, 9 dias úteis), com OIDC/PKCE, authlib 1.9.5, pytest 9.2.0, MFA obrigatória para operações sensíveis e administradores, e 3 histórias somando 11 pontos: o plano cabe com o Bruno (saldo de 14), mas sem ele o saldo cai para 9 e faltam 2 pontos, decisões que ainda estão pendentes junto com a confirmação do escopo de pessoa física.', 'similaridade': 0.756}

MEMÓRIA DE LONGO PRAZO DO USUÁRIO:
 - [semantica] O usuário prefere histórias com no máximo 5 pontos cada.
 - [semantica] O time do Portal usa pytest como framework de testes.
 - [episodica] Foi proposta a atualização da authlib 1.3.0 para 1.9.5 no outro serviço que ainda usa a 1.3.0, como história separada na sprint 13.
 - [semantica

## 4. Sessão 2: memória de longo prazo entre sessões

Um **novo** chatbot (nova sessão, memória de curto prazo vazia) para o mesmo usuário. Ele não tem o histórico da sessão 1 — só a memória de longo prazo, recuperada por busca vetorial.

In [11]:
bot2 = ChatbotTechLead(usuario="adriano", janela=4, callbacks=[log])
print("Mensagens na memória de curto prazo da nova sessão:", len(bot2.memoria_curta.chat_memory.messages))
mostrar(bot2.conversar("Bom dia! O que planejamos da última vez para o Portal?"))

Mensagens na memória de curto prazo da nova sessão: 0


[auditoria SQL] SELECT h.historia_id, h.epico, h.titulo, h.pontos, h.status, s.numero FROM historias h LEFT JOIN sprints s ON s.sprint_id=h.sprint_id JOIN projetos p ON p.projeto_id=h.projeto_id WHERE p.nome='Portal' AND (h.epico LIKE '%Google%' OR h.titulo LIKE '%Google%') LIMIT 50



🧑 Bom dia! O que planejamos da última vez para o Portal?
   intenção: consultar_backlog | estado da sessão: {'projeto': 'Portal'}
   memórias de longo prazo recuperadas: ['Em 05/11/2026, o épico Login com Google no Portal foi planejado para a sprint 13 (09/11 a 20/11, 9 dias úteis), com OIDC/PKCE, authlib 1.9.5, pytest 9.2.0, MFA obrigatória para operações sensíveis e administradores, e 3 histórias somando 11 pontos: o plano cabe com o Bruno (saldo de 14), mas sem ele o saldo cai para 9 e faltam 2 pontos, decisões que ainda estão pendentes junto com a confirmação do escopo de pessoa física.']
   ferramentas: ['consultar_backlog_sql']
🤖 Bom dia! Na última sessão (05/11/2026) planejamos o épico **Login com Google** no Portal, mas ele **ainda não está registrado no backlog**.

**O que foi planejado** (fonte: memória da sessão anterior)
- **Sprint:** 13, de 09/11 a 20/11, com 9 dias úteis.
- **Escopo:** 3 histórias, somando 11 pontos.
- **Técnica:** OIDC com PKCE, authlib 1.9.5 e pytest 9

In [12]:
# A preferência "histórias de até 5 pontos" foi dita na sessão 1; aqui ela precisa vir da memória semântica
mostrar(bot2.conversar("Planeje também a exportação do histórico de atendimentos em PDF para o Portal."))

[auditoria SQL] SELECT historia_id, titulo, pontos, status FROM historias WHERE projeto_id=(SELECT projeto_id FROM projetos WHERE nome='Portal') AND (titulo LIKE '%PDF%' OR titulo LIKE '%atendiment%' OR epico LIKE '%PDF%' OR epico LIKE '%atendiment%') LIMIT 50



🧑 Planeje também a exportação do histórico de atendimentos em PDF para o Portal.
   intenção: planejar_demanda | estado da sessão: {'projeto': 'Portal', 'sprint': 13, 'pacote': 'reportlab'}
   memórias de longo prazo recuperadas: ['(sessão mais recente) Em 05/11/2026, o épico Login com Google no Portal foi planejado para a sprint 13 (09/11 a 20/11, 9 dias úteis), com OIDC/PKCE, authlib 1.9.5, pytest 9.2.0, MFA obrigatória para operações sensíveis e administradores, e 3 histórias somando 11 pontos: o plano cabe com o Bruno (saldo de 14), mas sem ele o saldo cai para 9 e faltam 2 pontos, decisões que ainda estão pendentes junto com a confirmação do escopo de pessoa física.']
   ferramentas: ['calcular_capacidade_sprint', 'buscar_manual_engenharia', 'buscar_manual_engenharia', 'buscar_issues_github', 'buscar_issues_github', 'consultar_pypi', 'consultar_vulnerabilidades', 'consultar_backlog_sql', 'decompor_demanda']
🤖 # Plano: exportação do histórico de atendimentos em PDF (Portal)

**Con

### Atualização de memória (em vez de duplicação)

Se o usuário muda uma preferência, a memória antiga sobre o mesmo assunto é **atualizada** (o LLM decide ADICIONAR/ATUALIZAR entre as memórias mais parecidas):

In [13]:
mlp = MemoriaLongoPrazo(escopo="adriano")
print(mlp.lembrar("Adriano prefere histórias de no máximo 3 pontos.", "semantica"))
for m in mlp.recuperar("tamanho máximo das histórias preferido pelo Adriano", k=2):
    print(f" - sim={m.similaridade} [{m.tipo}] {m.conteudo}")

{'acao': 'atualizada', 'memoria_id': 87, 'antes': 'O usuário prefere histórias com no máximo 5 pontos cada.', 'depois': 'Adriano prefere histórias de no máximo 3 pontos.', 'similaridade': 0.532}
 - sim=0.69 [semantica] Adriano prefere histórias de no máximo 3 pontos.
 - sim=0.019 [episodica] (sessão mais recente) Para a exportação em PDF do Portal, a biblioteca escolhida é reportlab 5.1.0, a ser fixada na versão.


## 5. Fluxo com esclarecimento (entidade obrigatória ausente)

Um usuário novo, sem memória, pede uma demanda sem dizer o projeto. O fluxo de `planejar_demanda` exige o projeto: o chatbot pergunta, e o turno seguinte completa o estado e segue o fluxo.

In [14]:
MemoriaLongoPrazo(escopo="visitante").esquecer_tudo()
bot3 = ChatbotTechLead(usuario="visitante", callbacks=[log])
t = bot3.conversar("Quero adicionar modo escuro.")
print("pediu esclarecimento:", t.pediu_esclarecimento, "| ferramentas chamadas:", t.trajetoria)
mostrar(t)

pediu esclarecimento: True | ferramentas chamadas: []

🧑 Quero adicionar modo escuro.
   intenção: planejar_demanda | estado da sessão: {}
🤖 Para qual projeto é essa demanda: Portal, PagaFacil ou AppCampo?


In [15]:
mostrar(bot3.conversar("No AppCampo."))

[auditoria SQL] SELECT nome, descricao, stack, repositorio FROM projetos WHERE nome='AppCampo' LIMIT 50


[auditoria SQL] SELECT s.numero, s.inicio, s.fim, s.status FROM sprints s JOIN projetos p ON p.projeto_id=s.projeto_id WHERE p.nome='AppCampo' ORDER BY s.numero LIMIT 50



🧑 No AppCampo.
   intenção: planejar_demanda | estado da sessão: {'projeto': 'AppCampo', 'pacote': 'fastapi', 'sprint': 5}
   ferramentas: ['calcular_capacidade_sprint', 'buscar_manual_engenharia', 'buscar_issues_github', 'buscar_issues_github', 'consultar_backlog_sql', 'consultar_backlog_sql', 'consultar_pypi', 'calcular_capacidade_sprint', 'decompor_demanda']
🤖 **Conclusão:** o modo escuro soma 11 pontos em 3 histórias, e não cabe na sprint atual do AppCampo. A sprint 5 está com saldo de −5 pontos. Recomendo deixar as histórias no backlog para a próxima sprint. Ainda não existe sprint planejada, então não consegui calcular o saldo dela.

**Plano proposto (épico "Modo escuro no AppCampo")**

| # | História | Pontos |
|---|---|---|
| 1 | Alternar entre tema claro e escuro | 5 |
| 2 | Persistir a escolha de tema, com fallback para o tema do sistema | 3 |
| 3 | Sincronizar a preferência no Firebase entre dispositivos | 3 |
| | **Total** | **11** |

- Cada história tem critérios Gherkin 

## 6. Evidências e custo

In [16]:
r = log.resumo()
print(json.dumps({k: (v if k != "ferramentas" else len(v)) for k, v in r.items() if k != "erros"}, ensure_ascii=False, indent=1))
print("erros registrados:", len(r["erros"]))

{
 "chamadas_llm": 24,
 "tokens_entrada": 170577,
 "tokens_saida": 16632,
 "ferramentas": 31,
 "tempo_llm_s": 118.36,
 "tempo_ferramentas_s": 31.4
}
erros registrados: 0


## 7. Conclusões da fase

- **Contexto entre turnos:** o agente resolveu referências ("o plano", "a biblioteca que você sugeriu", "a sprint") pela memória de curto prazo e pelo estado de entidades, e decompôs pedidos compostos (simulação de capacidade + comparação com o plano; versão + vulnerabilidades + regra do MAN-04).
- **Compressão sem perda de decisão:** com janela de 4 mensagens, a preferência do turno 1 e o plano do turno 2 sobreviveram no resumo e foram usados no turno 5.
- **Entre sessões:** a sessão 2 recuperou o episódio e a preferência sem histórico de curto prazo.
- **Problemas encontrados e corrigidos durante a execução desta fase** (registrados no relatório):
  1. com os exemplos *few-shot* como pares de mensagens, o modelo os confundiu com turnos reais da conversa; os exemplos passaram para o system prompt, rotulados como ilustrativos;
  2. "a biblioteca que você sugeriu" disparava uma pergunta de esclarecimento, porque o pacote estava na resposta do agente e não no estado; o estado passou a absorver as entidades usadas nas ferramentas, e o classificador resolve referências com a última resposta;
  3. o extrator gravava regras do manual e números voláteis; o prompt de extração passou a excluí-los;
  4. uma preferência alterada ("3 pontos") era gravada ao lado da antiga ("5 pontos"); entrou a decisão ADICIONAR/ATUALIZAR por LLM;
  5. na sessão 2, "o que planejamos da última vez?" não recuperava o episódio (similaridade abaixo do limiar); entrou a recência episódica.
- **Contradição interna observada (caso de teste para a Fase 5):** numa execução, a resposta à simulação sem o Bruno começou com "sim, o plano ainda cabe" e terminou com "não cabe por 2 pontos". O verificador de coerência lógica da Fase 5 deve detectar esse tipo de falha.
- **Limitações:** a extração de memórias depende do LLM (pode gravar algo irrelevante ou deixar de gravar); a recuperação por similaridade pode trazer memórias antigas que já não valem (mitigado pela atualização por duplicata e pela data exibida em cada memória); o limiar de relevância (0,35) foi ajustado empiricamente para este modelo de embeddings.